In [ ]:
# Garantir que o diretório de trabalho é a raiz do projeto
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print('CWD:', os.getcwd())

In [ ]:
# Configuração dos parâmetros de execução
minitask_type = 'development_build_city'
device = 'cpu'
buffer_capacity = 5000
resume_from = 'experiments/checkpoints_long/dqn_ep50.pt'
start_episode = 50
checkpoint_dir = 'experiments/checkpoints_long'
print('Params:', device, minitask_type, buffer_capacity, resume_from, start_episode)

In [ ]:
# Criar diretório de checkpoints, se necessário
from pathlib import Path
Path(checkpoint_dir).mkdir(parents=True, exist_ok=True)
checkpoint_exists = Path(resume_from).exists()
print('Checkpoint existe?', checkpoint_exists, '|', resume_from)

In [ ]:
# Executar o script de treinamento (equivalente ao comando de terminal)
import sys, subprocess, shlex
cmd = [
    sys.executable, 'examples/train_dqn_long.py',
    '--device', device,
    '--minitask-type', minitask_type,
    '--buffer-capacity', str(buffer_capacity),
    '--resume-from', resume_from,
    '--start-episode', str(start_episode),
]
print('Running:', ' '.join(shlex.quote(str(x)) for x in cmd))
proc = subprocess.run(cmd, capture_output=False, check=False)
print('Return code:', proc.returncode)

In [ ]:
# Carregar logs e mostrar resumo
import pandas as pd
from pathlib import Path
csv_path = Path(checkpoint_dir) / 'training_logs.csv'
if csv_path.exists():
    df = pd.read_csv(csv_path)
    display(df.tail())
    if 'reward' in df.columns:
        print('Mean reward:', df['reward'].mean())
        print('Best reward:', df['reward'].max())
        print('Final epsilon:', df['epsilon'].iloc[-1] if 'epsilon' in df.columns else 'N/A')
else:
    print('Log não encontrado:', csv_path)
df_exists = 'df' in globals() and isinstance(df, pd.DataFrame) and not df.empty

In [ ]:
# Plot simples das recompensas por episódio (se disponível)
import matplotlib.pyplot as plt
if 'df_exists' in globals() and df_exists and 'reward' in df.columns:
    plt.figure(figsize=(10,3))
    plt.plot(df['reward'], label='reward')
    plt.xlabel('episode')
    plt.ylabel('reward')
    plt.title('Rewards por episódio')
    plt.legend()
    plt.show()
else:
    print('Nada para plotar (sem df ou coluna reward).')